In [2]:
import pandas as pd
import re

def gerar_dataset_balanceado(input_file, output_file, n_total_desejado=10000):
    dados_limpos = []
    
    # Abrimos o arquivo com tratamento de erro para caracteres especiais
    with open(input_file, 'r', encoding='utf-8', errors='ignore') as f:
        for linha in f:
            linha = linha.strip()
            if not linha:
                continue
            
            # EXPLICAÇÃO DO REGEX:
            # ^(\S+)      -> Pega o prefixo (tr, va, te...)
            # \s+         -> Pula qualquer espaço ou tab
            # (\w|\d)     -> Pega a CATEGORIA (seja Letra ou Dígito)
            # \s+         -> Pula o próximo espaço/tab
            # (.*)$       -> Pega todo o resto da frase
            match = re.match(r'^(\S+)\s+(\w|\d)\s+(.*)$', linha)
            
            if match:
                dados_limpos.append({
                    'prefixo': match.group(1),
                    'categoria': match.group(2),
                    'texto': match.group(3)
                })

    if not dados_limpos:
        print("Erro: O padrão das colunas não foi reconhecido. Verifique se há tabs/espaços entre tr e a categoria.")
        return

    df = pd.DataFrame(dados_limpos)

    # 2. Identificar categorias e calcular proporção
    categorias = df['categoria'].unique()
    n_categorias = len(categorias)
    amostras_por_cat = n_total_desejado // n_categorias
    
    print(f"Categorias detectadas: {categorias}")

    list_datasets = []
    for cat in categorias:
        subset = df[df['categoria'] == cat]
        
        # Se a categoria tiver menos que o necessário, pega tudo que tem
        n_a_pegar = min(len(subset), amostras_por_cat)
        list_datasets.append(subset.sample(n=n_a_pegar, random_state=42))
    
    # 3. Concatenar e embaralhar a ordem das linhas
    df_final = pd.concat(list_datasets).sample(frac=1, random_state=42)

    # 4. Salvar exatamente no formato original
    # Usamos o separador TAB (\t) e removemos aspas
    df_final.to_csv(output_file, sep='\t', header=False, index=False, quoting=3, escapechar=" ")
    
    print(f"---")
    print(f"Dataset gerado: {output_file}")
    print(f"Total de linhas: {len(df_final)}")
    print("Distribuição final:")
    print(df_final['categoria'].value_counts())

# Exemplo:
# gerar_dataset_balanceado_final('seu_arquivo.txt', 'saida_balanceada.txt', 10000)

In [3]:
for dataset in ["bigram_shift", "odd_man_out", "sentence_length"]:
    gerar_dataset_balanceado(f'../probing/data/raw/{dataset}.txt', f'{dataset}_2_000.txt', 2_000)

Categorias detectadas: ['O' 'I']
---
Dataset gerado: bigram_shift_2_000.txt
Total de linhas: 2000
Distribuição final:
categoria
I    1000
O    1000
Name: count, dtype: int64
Categorias detectadas: ['O' 'C']
---
Dataset gerado: odd_man_out_2_000.txt
Total de linhas: 2000
Distribuição final:
categoria
C    1000
O    1000
Name: count, dtype: int64
Categorias detectadas: ['0' '2' '5' '3' '1' '4']
---
Dataset gerado: sentence_length_2_000.txt
Total de linhas: 1998
Distribuição final:
categoria
0    333
2    333
3    333
4    333
1    333
5    333
Name: count, dtype: int64
